In [1]:
!pip install openai python-dotenv pymupdf

In [2]:
import json, os, re, sys
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(Path.cwd() / ".env")
api_key=os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

In [3]:
pdf_path=os.path.join(os.getcwd(),'samples/학칙.pdf')
pdf_path

'/Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/Practice/samples/학칙.pdf'

In [4]:
from pdf_summary import summarize_pdf,pdf_to_text


text_path=pdf_to_text(pdf_path)
print('text_path : ',text_path)
with open(text_path,'r',encoding='utf-8') as f:
  rules_text=f.read()

print('학칙 글자수',len(rules_text))

text_path :  /Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/Practice/samples/학칙.txt
학칙 글자수 47140


In [5]:
text=re.sub(r'\s+',' ',rules_text.strip())


In [6]:
def split_text(text,chunk_size=280,overlap=60):
  text=re.sub(r'\s+',' ',text.strip())
  if len(text)<=chunk_size:
    return [text]
  chunks,start=[],0
  while start<len(text):
    end=start+chunk_size
    chunks.append(text[start:end])
    if end>=len(text):
      break
    start=end-overlap
  return chunks
chunks=split_text(rules_text)
print('chunk수:',len(chunks))
print(chunks[0][:150])


chunk수: 213
Ⅰ. 대학원 학칙 / 7 Ⅰ. 대학원 학칙 제정: 1974. 05. 18. 개정(제122차): 2026. 06. 06. 제1장 총칙 제1조(목적) 대학원은 기독교 정신을 바탕으로 하여 창의적 이론과 과학적 방법을 탐 구하고, 지도적 인격을 도야하여 인류 문화 향상에 기


In [7]:
INDEX=[
  { 'chunk_id':f'RULE_C{i:03d}',
    'title':'연세대학교 대학원 학칙',
    'text':chunk,
  
  }
  for i,chunk in enumerate(chunks,1)
]
print('index chunk:',len(INDEX))

index chunk: 213


In [10]:
def tokenize(text):
  return {t.lower() for t in re.findall(r'[가-힣a-zA-Z0-9]+',text) if len(t)>1}

def search(query,top_k=3):
  q=tokenize(query)
  scored=[]
  for item in INDEX:
    score=len(q&tokenize(item['text']))
    if score>0:
      scored.append((score,item))
  scored.sort(key=lambda x:x[0],reverse=True)
  return [{'score':s,**it} for s,it in scored[:top_k]]

demo_question = '연세대학교에서 논문제출 시한을 연장한 학생은 휴학할 수 있는가? 예외는 무엇인가?'


hits=search(demo_question,top_k=3)

for h in hits:
  print(f"[{h['chunk_id']}] score={h['score']}")
  print(h['text'][:200], '...\n')

[RULE_C039] score=4
에 따라 제출하여 야 한다. ② 위 제1항의 시한에는 등록학기만 산입하며, 휴학 및 제적기간은 산입하지 아니한다. ③ 수료요건을 충족하고 합당한 사유가 있는 학생은 논문제출 시한 연장 사유서와 연구계획서를 제출하고 지도교수와 대학원장의 승인을 얻어 논문제출 시한을 2년 연 장할 수 있다. ④ 논문제출 시한 연장을 승인받은 기간에는 휴학할 수 없으나 다음 각 ...

[RULE_C040] score=3
 1. 입대휴학 2. 육아휴학 3. 그 외에 대학원장이 인정하는 중대한 사유에 의한 일반휴학 ⑤ 「장애인 등에 대한 특수교육법」 제15조 제1항의 특수교육대상자는 「대학원 학 칙」 제2조의8 제2항에 따라 합당한 사유가 있는 경우 대학원장의 승인을 받아 제3 항의 연장 시한을 예외로 한다. 제36조의2(논문제출) 학위논문 인준을 받은 학생은 대학원에서 정한 ...

[RULE_C038] score=2
다. 제35조(재심사) ① 학위논문 심사에 불합격 판정을 받은 학생은 적절한 수정 보완을 하여 재심사를 받을 수 있으며 이에 관한 사항은 따로 정한다. ② <삭제> 제36조(논문제출 시한) ① 석사학위과정 학생은 입학일로부터 4년 이내에, 박사학위과 정 학생은 입학일로부터 7년 이내에, 통합과정 학생은 입학일로부터 8년 이내에 학 위논문 심사에 합격하여 완 ...



In [12]:
def answer_with_rag(question, top_k=3):
    hits = search(question, top_k=top_k)
    if not hits:
        return {'question': question, 'answer': '관련 학칙 조항을 찾지 못했습니다.', 'sources': []}
    context = '\n\n'.join(f"[{h['chunk_id']}] {h['text']}" for h in hits)
    print("context:",context)
    r = client.chat.completions.create(
        model='gpt-4o-mini', temperature=0.1,
        messages=[
            {'role': 'system', 'content': 'Use only provided context from university regulations. Answer in Korean.'},
            {'role': 'user', 'content': f'질문: {question}\n\n학칙 문맥:\n{context}\n\n문맥에 없으면 모른다고 답하고 chunk_id를 bullet로 적으세요.'},
        ],
    )
    return {
        'question': question,
        'answer': r.choices[0].message.content,
        'sources': [h['chunk_id'] for h in hits],
    }

rag_result = answer_with_rag(demo_question)
print('=== RAG 사용 ===')
print(rag_result['answer'])
print('\n출처:', rag_result['sources'])

context: [RULE_C039] 에 따라 제출하여 야 한다. ② 위 제1항의 시한에는 등록학기만 산입하며, 휴학 및 제적기간은 산입하지 아니한다. ③ 수료요건을 충족하고 합당한 사유가 있는 학생은 논문제출 시한 연장 사유서와 연구계획서를 제출하고 지도교수와 대학원장의 승인을 얻어 논문제출 시한을 2년 연 장할 수 있다. ④ 논문제출 시한 연장을 승인받은 기간에는 휴학할 수 없으나 다음 각호의 휴학은 예외로 처리할 수 있다. 1. 입대휴학 2. 육아휴학 3. 그 외에 대학원장이 인정하는 중대한 사유에 의한 일반휴학 ⑤ 「장애인 등

[RULE_C040]  1. 입대휴학 2. 육아휴학 3. 그 외에 대학원장이 인정하는 중대한 사유에 의한 일반휴학 ⑤ 「장애인 등에 대한 특수교육법」 제15조 제1항의 특수교육대상자는 「대학원 학 칙」 제2조의8 제2항에 따라 합당한 사유가 있는 경우 대학원장의 승인을 받아 제3 항의 연장 시한을 예외로 한다. 제36조의2(논문제출) 학위논문 인준을 받은 학생은 대학원에서 정한 절차에 따라 완성 된 학위논문을 제출하여야 한다. 제8장 연구과정생, 외국인 학생, 공개강좌 제37조(연구과정생) 연구과정생은 대학원 강의를 

[RULE_C038] 다. 제35조(재심사) ① 학위논문 심사에 불합격 판정을 받은 학생은 적절한 수정 보완을 하여 재심사를 받을 수 있으며 이에 관한 사항은 따로 정한다. ② <삭제> 제36조(논문제출 시한) ① 석사학위과정 학생은 입학일로부터 4년 이내에, 박사학위과 정 학생은 입학일로부터 7년 이내에, 통합과정 학생은 입학일로부터 8년 이내에 학 위논문 심사에 합격하여 완성된 학위논문을 대학원에서 정한 절차에 따라 제출하여 야 한다. ② 위 제1항의 시한에는 등록학기만 산입하며, 휴학 및 제적기간은 산입하지 아니
=== RAG 사용 ===
연세대학교에서 논문제출 시한을 연장한 학생은 일반적으로 휴학할 수 없습니다. 그러나 다음과 같은 예외가 있습니다:

1. 입대휴학
2. 육아휴학
3. 대학원장이 인정하는 